<a href="https://colab.research.google.com/github/kousthub93/llm-systems-lab/blob/main/01-retrieval-benchmark/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% CELL 1 — Install deps (Colab)
!pip install -q datasets sentence-transformers faiss-cpu rank_bm25

In [ ]:

# %% CELL 2 — Load SQuAD v2 and build a passage corpus with ground truth
from datasets import load_dataset
import random

random.seed(42)

def load_corpus(n_contexts=3000):
    """
    SQuAD gives (question, context, answer) triples where many questions
    share the same context. We dedupe contexts into a passage corpus and
    keep a question -> correct_passage_id mapping as ground truth.
    """
    ds = load_dataset("rajpurkar/squad_v2", split="train")

    context_to_id = {}
    passages = []
    qa_pairs = []  # (question, correct_passage_id)

    for row in ds:
        ctx = row["context"]
        if ctx not in context_to_id:
            if len(passages) >= n_contexts:
                continue
            context_to_id[ctx] = f"p{len(passages)}"
            passages.append(ctx)

        if ctx in context_to_id and row["answers"]["text"]:  # skip unanswerable for v1
            qa_pairs.append((row["question"], context_to_id[ctx]))

    print(f"Loaded {len(passages)} passages, {len(qa_pairs)} labeled questions")
    return passages, qa_pairs

passages, qa_pairs = load_corpus(n_contexts=3000)

# Subsample questions for faster iteration during development
eval_questions = random.sample(qa_pairs, min(500, len(qa_pairs)))

README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

squad_v2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

squad_v2/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 1.35MB            

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Loaded 3000 passages, 13156 labeled questions


In [ ]:
# %% CELL 3 — Eval harness (build this BEFORE any retrieval variant — trust it, reuse it)
def recall_at_k(retrieved_ids, ground_truth_id, k):
    return 1 if ground_truth_id in retrieved_ids[:k] else 0

def reciprocal_rank(retrieved_ids, ground_truth_id):
    if ground_truth_id in retrieved_ids:
        return 1.0 / (retrieved_ids.index(ground_truth_id) + 1)
    return 0.0

def evaluate_retriever(retrieve_fn, eval_questions, k_values=(1, 3, 5, 10)):
    """
    retrieve_fn: function(question, top_k) -> list of passage_ids, ranked
    Returns a dict of aggregate metrics. This is what every config gets scored on.
    """
    import time
    results = {f"recall@{k}": [] for k in k_values}
    results["mrr"] = []
    latencies = []

    for question, gt_id in eval_questions:
        t0 = time.perf_counter()
        retrieved = retrieve_fn(question, top_k=max(k_values))
        latencies.append(time.perf_counter() - t0)

        for k in k_values:
            results[f"recall@{k}"].append(recall_at_k(retrieved, gt_id, k))
        results["mrr"].append(reciprocal_rank(retrieved, gt_id))

    summary = {metric: sum(vals) / len(vals) for metric, vals in results.items()}
    latencies.sort()
    summary["p50_latency_ms"] = latencies[len(latencies) // 2] * 1000
    summary["p95_latency_ms"] = latencies[int(len(latencies) * 0.95)] * 1000
    return summary

In [ ]:
# %% CELL 4 — Chunking strategies (start with "no chunking" baseline since SQuAD
# contexts are already paragraph-sized; add real chunking once corpus grows)
def chunk_fixed_size(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(" ".join(words[i:i + chunk_size]))
        i += chunk_size - overlap
    return chunks

def chunk_none(text):
    return [text]  # baseline: whole passage as one chunk

In [ ]:
# %% CELL 5 — Baseline: single embedding model + flat (exact) index
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("BAAI/bge-small-en-v1.5")  # start small/fast, swap later

def build_flat_index(passages, model):
    embeddings = model.encode(passages, normalize_embeddings=True, show_progress_bar=True)
    return embeddings  # cosine sim == dot product since normalized

passage_embeddings = build_flat_index(passages, model)
passage_ids = [f"p{i}" for i in range(len(passages))]

def flat_retrieve(question, top_k=10):
    q_emb = model.encode([question], normalize_embeddings=True)[0]
    scores = passage_embeddings @ q_emb  # dot product = cosine sim (normalized)
    top_idx = np.argsort(-scores)[:top_k]
    return [passage_ids[i] for i in top_idx]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

In [ ]:
# %% CELL 6 — Run the baseline and print the first row of your comparison table
baseline_results = evaluate_retriever(flat_retrieve, eval_questions)
print("Config: chunk=none | embed=bge-small | index=flat | retrieval=dense-only")
for metric, val in baseline_results.items():
    print(f"  {metric}: {val:.4f}")

# Next steps (don't build all at once — swap ONE variable at a time):
# 1. Swap embedding model -> BAAI/bge-large-en-v1.5, compare recall/latency
# 2. Add BM25 sparse retrieval (rank_bm25) as a second retrieve_fn
# 3. Implement reciprocal rank fusion combining dense + sparse rankings
# 4. Swap flat index -> faiss HNSW, sweep ef_search, compare recall vs. latency
# 5. Log every config's `evaluate_retriever` output into one comparison table

Config: chunk=none | embed=bge-small | index=flat | retrieval=dense-only
  recall@1: 0.6520
  recall@3: 0.8240
  recall@5: 0.8840
  recall@10: 0.9280
  mrr: 0.7493
  p50_latency_ms: 25.2670
  p95_latency_ms: 57.1852


In [ ]:
# %% CELL 7 — BM25 sparse retrieval (compare against the dense baseline)
from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    # BM25 works on token overlap, not semantics -- lowercase + split is enough here
    return text.lower().split()

# Build the BM25 index once (like passage_embeddings was your dense "index")
tokenized_passages = [simple_tokenize(p) for p in passages]
bm25 = BM25Okapi(tokenized_passages)

def bm25_retrieve(question, top_k=10):
    tokenized_query = simple_tokenize(question)
    scores = bm25.get_scores(tokenized_query)  # one score per passage
    top_idx = np.argsort(-scores)[:top_k]
    return [passage_ids[i] for i in top_idx]

bm25_results = evaluate_retriever(bm25_retrieve, eval_questions)
print("Config: chunk=none | retrieval=BM25-only (sparse)")
for metric, val in bm25_results.items():
    print(f"  {metric}: {val:.4f}")

print("\n--- Comparison: dense vs sparse ---")
for metric in baseline_results:
    dense_val = baseline_results[metric]
    sparse_val = bm25_results[metric]
    print(f"  {metric:20s} dense={dense_val:.4f}  bm25={sparse_val:.4f}  diff={sparse_val - dense_val:+.4f}")

Config: chunk=none | retrieval=BM25-only (sparse)
  recall@1: 0.5060
  recall@3: 0.6420
  recall@5: 0.7040
  recall@10: 0.7660
  mrr: 0.5866
  p50_latency_ms: 7.7703
  p95_latency_ms: 13.3469

--- Comparison: dense vs sparse ---
  recall@1             dense=0.6520  bm25=0.5060  diff=-0.1460
  recall@3             dense=0.8240  bm25=0.6420  diff=-0.1820
  recall@5             dense=0.8840  bm25=0.7040  diff=-0.1800
  recall@10            dense=0.9280  bm25=0.7660  diff=-0.1620
  mrr                  dense=0.7493  bm25=0.5866  diff=-0.1627
  p50_latency_ms       dense=25.2670  bm25=7.7703  diff=-17.4967
  p95_latency_ms       dense=57.1852  bm25=13.3469  diff=-43.8384


In [ ]:
# %% CELL 8 — Reciprocal Rank Fusion: combine dense + BM25 rankings
def rrf_retrieve(question, top_k=10, k_constant=60):
    """
    RRF combines two ranked lists without needing their scores to be on the
    same scale (dense similarity and BM25 scores aren't comparable directly).
    Each passage's RRF score = sum of 1/(k_constant + rank) across all lists
    it appears in. Passages that rank well in BOTH lists rise to the top.
    """
    dense_ranked = flat_retrieve(question, top_k=50)   # pull a wider pool to fuse
    sparse_ranked = bm25_retrieve(question, top_k=50)

    rrf_scores = {}
    for rank, pid in enumerate(dense_ranked):
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1.0 / (k_constant + rank + 1)
    for rank, pid in enumerate(sparse_ranked):
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1.0 / (k_constant + rank + 1)

    ranked_ids = sorted(rrf_scores.keys(), key=lambda pid: -rrf_scores[pid])
    return ranked_ids[:top_k]

rrf_results = evaluate_retriever(rrf_retrieve, eval_questions)
print("Config: chunk=none | retrieval=hybrid-RRF (dense+bm25)")
for metric, val in rrf_results.items():
    print(f"  {metric}: {val:.4f}")

print("\n--- Full comparison table ---")
print(f"{'metric':20s} {'dense':>8s} {'bm25':>8s} {'hybrid-rrf':>12s}")
for metric in baseline_results:
    print(f"{metric:20s} {baseline_results[metric]:8.4f} {bm25_results[metric]:8.4f} {rrf_results[metric]:12.4f}")

Config: chunk=none | retrieval=hybrid-RRF (dense+bm25)
  recall@1: 0.6460
  recall@3: 0.8080
  recall@5: 0.8580
  recall@10: 0.9160
  mrr: 0.7354
  p50_latency_ms: 27.4301
  p95_latency_ms: 39.9803

--- Full comparison table ---
metric                  dense     bm25   hybrid-rrf
recall@1               0.6520   0.5060       0.6460
recall@3               0.8240   0.6420       0.8080
recall@5               0.8840   0.7040       0.8580
recall@10              0.9280   0.7660       0.9160
mrr                    0.7493   0.5866       0.7354
p50_latency_ms        25.2670   7.7703      27.4301
p95_latency_ms        57.1852  13.3469      39.9803


In [ ]:
# %% CELL 9 — BM25 v2 (better tokenizer) + dense/BM25 overlap analysis
import re

STOPWORDS = {"the","a","an","of","in","on","at","to","is","was","were","are","and","or",
             "for","by","with","as","what","which","who","when","where","how","did","does",
             "do","from","that","this","it","its","be","been","has","have","had"}

def better_tokenize(text):
    return [t for t in re.findall(r"\w+", text.lower()) if t not in STOPWORDS]

bm25_v2 = BM25Okapi([better_tokenize(p) for p in passages])

def bm25_v2_retrieve(question, top_k=10):
    scores = bm25_v2.get_scores(better_tokenize(question))
    top_idx = np.argsort(-scores)[:top_k]
    return [passage_ids[i] for i in top_idx]

bm25_v2_results = evaluate_retriever(bm25_v2_retrieve, eval_questions)
print(f"{'metric':20s} {'bm25 v1':>9s} {'bm25 v2':>9s}")
for m in ["recall@1", "recall@5", "recall@10", "mrr"]:
    print(f"{m:20s} {bm25_results[m]:9.4f} {bm25_v2_results[m]:9.4f}")

# Who finds what? (top-5 hits)
both = dense_only = bm25_only = neither = 0
for q, gt in eval_questions:
    d = gt in flat_retrieve(q, top_k=5)
    b = gt in bm25_v2_retrieve(q, top_k=5)
    if d and b: both += 1
    elif d: dense_only += 1
    elif b: bm25_only += 1
    else: neither += 1
print(f"\nTop-5 overlap over {len(eval_questions)} questions:")
print(f"  both found it:   {both}")
print(f"  dense only:      {dense_only}")
print(f"  bm25 only:       {bm25_only}   <- the most hybrid can add")
print(f"  neither:         {neither}")

metric                 bm25 v1   bm25 v2
recall@1                0.5060    0.6820
recall@5                0.7040    0.8860
recall@10               0.7660    0.9300
mrr                     0.5866    0.7667

Top-5 overlap over 500 questions:
  both found it:   398
  dense only:      44
  bm25 only:       45   <- the most hybrid can add
  neither:         13


In [ ]:
# =============================================================================
# Project 1, part 2. Needs Cells 1-9 already run in this Colab session
# (uses: passages, passage_ids, eval_questions, evaluate_retriever, flat_retrieve,
#  bm25_retrieve, bm25_v2_retrieve, baseline_results, bm25_v2_results, np, SentenceTransformer)
# If the runtime restarted: Runtime -> Run all on Cells 1-9 first.
# =============================================================================

# %% CELL 10 -- Hybrid RRF using the fixed BM25 (v2), plus weighted fusion
def rrf_fuse(ranked_lists, weights, k_constant=60, top_k=10):
    """Weighted reciprocal rank fusion: score(p) = sum_i  w_i / (k + rank_i(p))."""
    scores = {}
    for ranked, w in zip(ranked_lists, weights):
        for rank, pid in enumerate(ranked):
            scores[pid] = scores.get(pid, 0.0) + w / (k_constant + rank + 1)
    return sorted(scores, key=lambda pid: -scores[pid])[:top_k]

def make_hybrid(dense_fn, sparse_fn, w_dense=1.0, w_sparse=1.0, pool=50):
    """Build a retrieve_fn that fuses a dense and a sparse retriever."""
    def retrieve(question, top_k=10):
        dense_ranked = dense_fn(question, top_k=pool)
        sparse_ranked = sparse_fn(question, top_k=pool)
        return rrf_fuse([dense_ranked, sparse_ranked], [w_dense, w_sparse], top_k=top_k)
    return retrieve

hybrid_small_variants = {
    "hybrid small+bm25v2 (1:1)":      make_hybrid(flat_retrieve, bm25_v2_retrieve, 1.0, 1.0),
    "hybrid small+bm25v2 (dense 2:1)": make_hybrid(flat_retrieve, bm25_v2_retrieve, 2.0, 1.0),
    "hybrid small+bm25v2 (bm25 2:1)":  make_hybrid(flat_retrieve, bm25_v2_retrieve, 1.0, 2.0),
}

def print_row(name, res):
    print(f"{name:34s} r@1={res['recall@1']:.3f}  r@5={res['recall@5']:.3f}  "
          f"r@10={res['recall@10']:.3f}  mrr={res['mrr']:.3f}")

print("Reference rows from earlier cells:")
print_row("dense bge-small", baseline_results)
print_row("bm25 v2", bm25_v2_results)
print("\nHybrid with BM25 v2:")
for name, fn in hybrid_small_variants.items():
    print_row(name, evaluate_retriever(fn, eval_questions))

Reference rows from earlier cells:
dense bge-small                    r@1=0.652  r@5=0.884  r@10=0.928  mrr=0.749
bm25 v2                            r@1=0.682  r@5=0.886  r@10=0.930  mrr=0.767

Hybrid with BM25 v2:
hybrid small+bm25v2 (1:1)          r@1=0.734  r@5=0.938  r@10=0.966  mrr=0.819
hybrid small+bm25v2 (dense 2:1)    r@1=0.738  r@5=0.930  r@10=0.960  mrr=0.818
hybrid small+bm25v2 (bm25 2:1)     r@1=0.714  r@5=0.928  r@10=0.964  mrr=0.809


In [ ]:
# %% CELL 11 -- Swap the dense model: bge-small -> bge-large (one variable at a time)
import time

large_model = SentenceTransformer("BAAI/bge-large-en-v1.5")
t0 = time.perf_counter()
large_embeddings = large_model.encode(
    passages, normalize_embeddings=True, batch_size=32, show_progress_bar=True
)
print(f"Embedded {len(passages)} passages in {time.perf_counter() - t0:.0f}s, "
      f"shape={large_embeddings.shape}")  # expect (3000, 1024)

def make_dense_retriever(st_model, embeddings, query_prefix=""):
    def retrieve(question, top_k=10):
        q_emb = st_model.encode([query_prefix + question], normalize_embeddings=True)[0]
        scores = embeddings @ q_emb
        top_idx = np.argsort(-scores)[:top_k]
        return [passage_ids[i] for i in top_idx]
    return retrieve

# Same query handling as bge-small (no prefix) so the model size is the only change.
dense_large = make_dense_retriever(large_model, large_embeddings)
# BGE's recommended instruction for short-query -> passage retrieval, as a separate row.
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
dense_large_prefix = make_dense_retriever(large_model, large_embeddings, BGE_QUERY_PREFIX)
hybrid_large = make_hybrid(dense_large, bm25_v2_retrieve, 1.0, 1.0)

print("\nbge-large rows:")
print_row("dense bge-small (reference)", baseline_results)
print_row("dense bge-large", evaluate_retriever(dense_large, eval_questions))
print_row("dense bge-large + query prefix", evaluate_retriever(dense_large_prefix, eval_questions))
print_row("hybrid large+bm25v2 (1:1)", evaluate_retriever(hybrid_large, eval_questions))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Embedded 3000 passages in 93s, shape=(3000, 1024)

bge-large rows:
dense bge-small (reference)        r@1=0.652  r@5=0.884  r@10=0.928  mrr=0.749
dense bge-large                    r@1=0.704  r@5=0.912  r@10=0.972  mrr=0.795
dense bge-large + query prefix     r@1=0.698  r@5=0.924  r@10=0.962  mrr=0.788
hybrid large+bm25v2 (1:1)          r@1=0.742  r@5=0.950  r@10=0.976  mrr=0.834


In [ ]:
# %% CELL 12 -- Final table: every config re-run and timed in ONE pass
import pandas as pd

configs = {
    "dense bge-small": flat_retrieve,
    "bm25 v1": bm25_retrieve,
    "bm25 v2": bm25_v2_retrieve,
    **hybrid_small_variants,
    "dense bge-large": dense_large,
    "dense bge-large + query prefix": dense_large_prefix,
    "hybrid large+bm25v2 (1:1)": hybrid_large,
}

# Warm-up so the first config doesn't pay GPU start-up cost in its latency numbers
for fn in configs.values():
    for q, _ in eval_questions[:3]:
        fn(q, top_k=10)

final = {name: evaluate_retriever(fn, eval_questions) for name, fn in configs.items()}
df = pd.DataFrame(final).T[["recall@1", "recall@3", "recall@5", "recall@10", "mrr",
                            "p50_latency_ms", "p95_latency_ms"]]
print(df.round(3).to_string())
df.round(4).to_csv("project1_results.csv")
print("\nSaved project1_results.csv (download it from the Colab file panel on the left).")

                                 recall@1  recall@3  recall@5  recall@10    mrr  p50_latency_ms  p95_latency_ms
dense bge-small                     0.652     0.824     0.884      0.928  0.749          14.082          17.400
bm25 v1                             0.506     0.642     0.704      0.766  0.587           8.085          13.085
bm25 v2                             0.682     0.828     0.886      0.930  0.767           4.218           7.310
hybrid small+bm25v2 (1:1)           0.734     0.894     0.938      0.966  0.819          21.638          29.110
hybrid small+bm25v2 (dense 2:1)     0.738     0.890     0.930      0.960  0.818          21.622          29.401
hybrid small+bm25v2 (bm25 2:1)      0.714     0.882     0.928      0.964  0.809          21.351          28.998
dense bge-large                     0.704     0.870     0.912      0.972  0.795          27.422          38.646
dense bge-large + query prefix      0.698     0.862     0.924      0.962  0.788          26.517         